# EEG 模型：pooled vs personal 现场快速测试

这个 Notebook 只需要一个 EDF 路径，就会复用仓库当前旧模型和 personal model 的正式推理链：

```text
EDF → MNE 读取/EEG 通道选择/必要时重采样
    → 0.5–43 Hz FIR → 4 s 窗口、2 s 步长
    → Welch 频带特征 → 各模型内置的 StandardScaler → PCA → RBF SVC
    → pooled/personal 窗口预测、类别数量/比例和可用时的准确率
```

重要边界：本 Notebook 只做 inference，不训练、不 fit、不微调、不做个体校准，也不写入正式 artifacts。文件名真值只用于评估，不参与模型输入；zqd 或无法确认身份时只运行 pooled，并明确跳过 personal。

## 使用方法

1. 执行导入和配置单元格。
2. 在 `EDF_PATH` 中填写路径；也可以在本地桌面环境执行 `choose_edf_file()` 后选择 EDF。
3. 按顺序执行代码单元格，最后一个单元格会显示 pooled vs personal 的窗口级预测和汇总。

默认路径是仓库中一个已存在、符合 `被试_状态_时间戳.edf` 字段顺序的示例：`lyc_focus_202609072034_raw.edf`。它可以直接替换成现场文件。`focus1` 等状态编号和 `_raw` 等额外后缀会被严格解析；subject 只接受明确的 `lyc`、`zyf`、`zqd`。

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

# Notebook 可能从仓库根目录或 notebooks/ 目录启动。向上查找正式脚本，
# 这样不依赖用户当前工作目录，也不复制一套 EEG 处理代码。
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "scripts" / "eeg_pipeline_utils.py").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("无法定位 EEGAttention 仓库根目录")
    REPO_ROOT = REPO_ROOT.parent

SCRIPTS_DIR = REPO_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

# 这两个 import 是复用正式实现的关键：
# eeg_utils 提供 MNE/预处理/窗口/Welch 特征；baseline 提供冻结协议常量。
import eeg_pipeline_utils as eeg_utils
import legacy_baseline_v0 as baseline
from subject_model_utils import parse_edf_identity, run_quick_comparison
MODEL_LABELS = tuple(baseline.LABELS)

ARTIFACT_DIR = REPO_ROOT / "artifacts" / baseline.BASELINE_VERSION
PIPELINE_PATH = ARTIFACT_DIR / "pipeline.joblib"
CONFIG_PATH = ARTIFACT_DIR / "config.json"
FREEZE_MANIFEST_PATH = ARTIFACT_DIR / "freeze_manifest.json"

print(f"仓库根目录: {REPO_ROOT}")
print(f"冻结模型: {PIPELINE_PATH}")
print(f"Personal models: {REPO_ROOT / 'artifacts' / 'subject_models'}")

## 1. 只填写或选择一个 EDF 路径

输入文件名优先遵循 `subject_status_timestamp.edf`。下面的选择器是可选便利功能；核心输入始终是一个路径字符串，不需要 CSV 或额外元数据文件。

In [ ]:
# 现场使用时把这个字符串替换成 EDF 路径。相对路径相对于仓库根目录解析。
EDF_PATH = r"C:\Users\13313\Desktop\lyc_unfocus_202609072312.edf"
def choose_edf_file(initial_dir: Path | None = None) -> str:
    """在本地桌面 Jupyter 中弹出文件选择器；无 GUI 时给出可读提示。"""
    try:
        from tkinter import Tk, filedialog

        root = Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        selected = filedialog.askopenfilename(
            initialdir=str(initial_dir or REPO_ROOT / "data"),
            title="选择一个 EDF 文件",
            filetypes=[("EDF files", "*.edf"), ("All files", "*.*")],
        )
        root.destroy()
        return selected
    except Exception as exc:
        print(f"当前环境无法打开图形文件选择器：{exc}")
        print("请直接在 EDF_PATH 中填写路径。")
        return ""

# 如果希望选择文件，可以取消下一行注释：
# EDF_PATH = choose_edf_file()
print(f"待测 EDF: {EDF_PATH or '尚未填写'}")

## 2. 严格解析 EDF 文件名中的标签

不使用模糊的 `"focus" in filename` 判断，而是读取 `Path(path).stem`，再用 `_` 分割：

```text
lyc_focus_202609072034_raw
│   │      │              │
│   │      │              └─ 额外后缀，可存在
│   │      └─ timestamp
│   └─ status 字段
└─ subject 字段
```

`focus`、`unfocus` 可直接作为模型任务真值；`focus1` 等编号状态也会被严格解析；`iu`、`ou` 按正式 baseline 的 `LABEL_MAPPING` 合并到 `unfocus`。`daze`、`rest`、`death` 不是当前二分类真值。只有明确的 `lyc`/`zyf`/`zqd` subject 才被识别；zqd 与未知 subject 均跳过 personal model。

In [ ]:
# 身份解析集中在 subject_model_utils；只接受明确的 lyc/zyf/zqd，
# 不根据 EEG 内容猜测 subject，也不在 Notebook 里复制解析规则。
def parse_edf_filename(edf_path: Path) -> dict[str, object]:
    parsed = parse_edf_identity(edf_path)
    for warning in parsed["warnings"]:
        print(f"⚠️ {warning} 文件名: {edf_path.name}")
    if not parsed["warnings"]:
        print(
            f"文件名真值: subject={parsed['subject']}, "
            f"raw_status={parsed['raw_status']}, true_label={parsed['true_label']}, "
            f"timestamp={parsed['timestamp']}"
        )
    return parsed

## 3. 加载 pooled 与 personal 模型

模型文件已经包含训练后的 StandardScaler、PCA 和 SVC 状态。这里做完整性与结构检查，但绝不调用 `fit()`；Notebook 后续只会调用 `predict()`。lyc/zyf 只加载与文件名 subject 对应的 personal model，zqd/unknown 明确只保留 pooled。

In [ ]:
def load_frozen_pipeline():
    """读取并验证 legacy_baseline_v0 的冻结 Pipeline。"""
    for path in (PIPELINE_PATH, CONFIG_PATH, FREEZE_MANIFEST_PATH):
        if not path.exists():
            raise FileNotFoundError(f"缺少冻结模型文件: {path}")

    config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    freeze_manifest = json.loads(FREEZE_MANIFEST_PATH.read_text(encoding="utf-8"))
    for name, path in {"pipeline.joblib": PIPELINE_PATH, "config.json": CONFIG_PATH}.items():
        expected = freeze_manifest.get("artifact_sha256", {}).get(name)
        if expected and eeg_utils.sha256_file(path) != expected:
            raise AssertionError(f"冻结文件 SHA-256 不匹配: {name}")

    pipeline = joblib.load(PIPELINE_PATH)
    if list(pipeline.named_steps) != ["scaler", "pca", "svc"]:
        raise AssertionError(f"非预期的 Pipeline 步骤: {list(pipeline.named_steps)}")
    if not all(hasattr(pipeline.named_steps[name], attr) for name, attr in (("scaler", "mean_"), ("pca", "components_"), ("svc", "support_"))):
        raise AssertionError("冻结 Pipeline 看起来尚未 fit")
    if int(pipeline.named_steps["scaler"].n_features_in_) != 240:
        raise AssertionError("冻结模型不是当前 240 维特征协议")
    if config.get("labels") != list(MODEL_LABELS):
        raise AssertionError("config.json 的标签顺序与正式 baseline 不一致")

    print(f"已加载冻结模型: {PIPELINE_PATH.name}")
    print(f"Pipeline: {list(pipeline.named_steps)}; features: {pipeline.named_steps['scaler'].n_features_in_}")
    return pipeline, config

PIPELINE, MODEL_CONFIG = load_frozen_pipeline()

## 4. 读取 EDF、复用正式特征链并输出结果

单个现场 EDF 没有 manifest 的 `activity_start_s`/`activity_end_s` 时，这里把整个 EDF 作为一个逻辑 segment（`0` 到 recording duration）。预处理和特征计算本身仍全部调用正式工具函数；因此不存在 Notebook 自己实现的第二套滤波、窗口或频带逻辑。

函数只调用共享的 EDF/预处理/特征函数一次，再把同一特征矩阵送入 pooled 和适用的 personal model；输出文件名、subject、true label、每个模型的准确率、预测类别数量/比例和 pooled vs personal 对比。

In [ ]:
def run_quick_test(edf_path: str | Path) -> dict[str, object]:
    """复用共享推理辅助，比较 pooled 与适用的 personal model。"""
    return run_quick_comparison(edf_path, REPO_ROOT)

## 5. 执行一次现场测试

最后一个单元格是唯一需要反复执行的地方：换 `EDF_PATH` 后重新执行即可。函数只返回内存中的结果，不把 CSV、JSON 或模型写回仓库。

In [ ]:
if not EDF_PATH:
    print("请先在上方填写 EDF_PATH，或取消 choose_edf_file() 注释选择文件。")
else:
    RESULT = run_quick_test(EDF_PATH)
    print("\n已完成：RESULT['predictions'] 按模型保存窗口级预测，RESULT['distributions'] 保存类别数量/比例，RESULT['model_summary'] 保存 pooled vs personal 对比。")